In [1]:
require 'gd'

# this function show the image on the notebook
def render(file_name)
    IRuby.display "<img src='#{file_name}' />", mime: "text/html"
end

:render

In [4]:


# ═════════════════════════════════════════════════════════════════
#  SafeMath — expression evaluator using Ruby's Math module
# ═════════════════════════════════════════════════════════════════
module SafeMath
  class EvalContext
    include Math
    def pi; Math::PI; end
    def e;  Math::E;  end

    def initialize(vars)
      vars.each { |k, v| val = v.to_f; define_singleton_method(k) { val } }
    end

    def ln(v);    Math.log(v);   end
    def log(v);   Math.log(v);   end
    def log10(v); Math.log10(v); end
    def log2(v);  Math.log2(v);  end
    def abs(v);   v.abs;         end
    def cbrt(v);  Math.cbrt(v);  end

    def eval_expr(expr)
      result = eval(expr)
      return nil unless result.is_a?(Numeric)
      return nil if result.respond_to?(:infinite?) && result.infinite?
      return nil if result.respond_to?(:nan?)      && result.nan?
      result.to_f
    end
  end

  def self.evaluate(expr, vars = {})
    EvalContext.new(vars).eval_expr(expr)
  rescue Math::DomainError, ZeroDivisionError, FloatDomainError, StandardError
    nil
  end
end


# ═════════════════════════════════════════════════════════════════
#  Plot3D — perspective 3D plotter built on ruby-libgd
#
#  Methods:
#    .surface(expr)   z = f(x,y) — viridis colormap shading
#    .bars(data)      3D bar chart — flat face shading + axes + labels
#    .render(path)    saves PNG and calls render()
#
#  All constructor options have sensible defaults — just call Plot3D.new
# ═════════════════════════════════════════════════════════════════
class Plot3D

  VIRIDIS = [
    [ 68,   1,  84], [ 72,  35, 116], [ 64,  67, 135],
    [ 52,  94, 141], [ 41, 120, 142], [ 32, 144, 140],
    [ 34, 167, 132], [ 68, 190, 112], [121, 209,  81],
    [189, 222,  38], [253, 231,  37],
  ].freeze

  BAR_PALETTE = [
    [ 78, 121, 167], [242, 142,  43], [ 89, 161,  79],
    [237,  88,  84], [176, 122, 161], [255, 157, 167],
    [118, 183, 178], [156, 117,  95], [186, 176, 171],
  ].freeze

  def initialize(
    width:      700,
    height:     520,
    xmin:      -5.0, xmax:  5.0,
    ymin:      -5.0, ymax:  5.0,
    zmin:       nil, zmax:  nil,
    resolution: 40,
    yaw:        35.0,
    pitch:      30.0,
    fov:        480.0,
    cam_dist:   18.0,
    bg:         [250, 250, 248]
  )
    @width      = width
    @height     = height
    @xmin       = xmin.to_f;  @xmax     = xmax.to_f
    @ymin       = ymin.to_f;  @ymax     = ymax.to_f
    @zmin_cfg   = zmin&.to_f; @zmax_cfg = zmax&.to_f
    @resolution = resolution
    @fov        = fov.to_f
    @cam_dist   = cam_dist.to_f
    @bg         = bg

    yr = yaw   * Math::PI / 180.0
    pr = pitch * Math::PI / 180.0
    @cos_y = Math.cos(yr); @sin_y = Math.sin(yr)
    @cos_p = Math.cos(pr); @sin_p = Math.sin(pr)

    @mode = nil
    @expr = nil
    @data = nil
  end

  # ── Public API ───────────────────────────────────────────────

  def surface(expr)
    @mode = :surface
    @expr = expr
    self
  end

  def bars(data)
    @mode = :bars
    @data = data
    self
  end

  def render(path = "/work/graph.png")
    img = case @mode
          when :surface then render_surface
          when :bars    then render_bars
          else raise "Call .surface(expr) or .bars(data) before .render"
          end
    img.save("/work/#{path}")
  end

  private

  # ════════════════════════════════════════════════════════════
  #  SURFACE RENDERING
  # ════════════════════════════════════════════════════════════
  def render_surface
    img  = new_image
    grid = sample_grid
    nx   = @resolution
    ny   = @resolution

    zs     = grid.flatten.compact
    zmin   = @zmin_cfg || zs.min
    zmax   = @zmax_cfg || zs.max
    zrange = (zmax - zmin).nonzero? || 1.0

    patches = []
    nx.times do |i|
      ny.times do |j|
        z00 = grid[i][j];         next unless z00
        z10 = grid[i+1]&.[](j);   next unless z10
        z01 = grid[i][j+1];       next unless z01
        z11 = grid[i+1]&.[](j+1); next unless z11

        x0 = @xmin + i.to_f       / nx * (@xmax - @xmin)
        x1 = @xmin + (i+1).to_f   / nx * (@xmax - @xmin)
        y0 = @ymin + j.to_f       / ny * (@ymax - @ymin)
        y1 = @ymin + (j+1).to_f   / ny * (@ymax - @ymin)

        pts3d    = [[x0,y0,z00],[x1,y0,z10],[x1,y1,z11],[x0,y1,z01]]
        avg_z    = (z00 + z10 + z01 + z11) / 4.0
        proj     = pts3d.map { |p| rotate3d(*p) }
        avg_dep  = proj.sum { |p| p[2] } / 4.0

        patches << { proj: proj, avg_z: avg_z, depth: avg_dep }
      end
    end

    patches.sort_by! { |p| -p[:depth] }

    patches.each do |patch|
      t     = (patch[:avg_z] - zmin) / zrange
      color = colormap(t)
      c     = gd_color(*color)
      pts2d = patch[:proj].map { |p| project2d(*p) }
      draw_filled_polygon(img, pts2d, c)
      draw_polygon_outline(img, pts2d, gd_color(*darken(color, 0.65)))
    end

    img
  end

  # ════════════════════════════════════════════════════════════
  #  BAR CHART RENDERING
  # ════════════════════════════════════════════════════════════
  def render_bars
    img    = new_image
    labels = @data.keys
    values = @data.values.map(&:to_f)
    n      = labels.size

    vmax   = (@zmax_cfg || values.max).to_f
    vmax   = 1.0 if vmax.zero?

    bar_w  = 0.55
    bar_d  = 0.55
    gap    = 1.4
    total  = (n - 1) * gap
    x0_base = -total / 2.0

    # ── Draw floor plane ───────────────────────────────────────
    floor_y_range = bar_d * 1.5
    fx0 = x0_base - bar_w; fx1 = x0_base + (n - 1) * gap + bar_w
    fy0 = -floor_y_range;  fy1 =  floor_y_range
    floor_pts = [[fx0,fy0,0],[fx1,fy0,0],[fx1,fy1,0],[fx0,fy1,0]]
      .map { |p| project2d(*rotate3d(*p)) }
    draw_filled_polygon(img, floor_pts, gd_color(235, 235, 232))
    draw_polygon_outline(img, floor_pts, gd_color(200, 200, 195))

    # ── Draw axes ──────────────────────────────────────────────
    draw_axes(img, x0_base, n, gap, bar_w, vmax)

    # ── Draw bars ──────────────────────────────────────────────
    # Sort bars back-to-front by their projected depth
    bar_items = n.times.map do |i|
      cx    = x0_base + i * gap
      cy    = 0.0
      val   = values[i]
      zh    = (val / vmax) * 7.5

      x0 = cx - bar_w/2.0; x1 = cx + bar_w/2.0
      y0 = cy - bar_d/2.0; y1 = cy + bar_d/2.0

      center_proj = rotate3d(cx, cy, zh / 2.0)
      { i: i, cx: cx, cy: cy, val: val, zh: zh,
        x0: x0, x1: x1, y0: y0, y1: y1, depth: center_proj[2] }
    end
    bar_items.sort_by! { |b| -b[:depth] }

    bar_items.each do |b|
      i  = b[:i]
      x0 = b[:x0]; x1 = b[:x1]
      y0 = b[:y0]; y1 = b[:y1]
      z0 = 0.0;    z1 = b[:zh]
      base_color = BAR_PALETTE[i % BAR_PALETTE.size]

      faces = [
        { pts: [[x0,y0,z0],[x1,y0,z0],[x1,y0,z1],[x0,y0,z1]], shade: 0.82 },  # front
        { pts: [[x1,y0,z0],[x1,y1,z0],[x1,y1,z1],[x1,y0,z1]], shade: 0.62 },  # right
        { pts: [[x0,y0,z1],[x1,y0,z1],[x1,y1,z1],[x0,y1,z1]], shade: 1.00 },  # top
      ]

      faces_proj = faces.map do |face|
        proj    = face[:pts].map { |p| rotate3d(*p) }
        avg_dep = proj.sum { |p| p[2] } / proj.size.to_f
        pts2d   = proj.map { |p| project2d(*p) }
        { pts2d: pts2d, shade: face[:shade], depth: avg_dep }
      end
      faces_proj.sort_by! { |f| -f[:depth] }

      faces_proj.each do |face|
        shaded = shade_color(base_color, face[:shade])
        draw_filled_polygon(img, face[:pts2d], gd_color(*shaded))
        draw_polygon_outline(img, face[:pts2d], gd_color(*darken(shaded, 0.72)))
      end

      # Value label drawn above top-face center
      top_pts = faces.find { |f| f[:shade] == 1.00 }[:pts]
        .map { |p| project2d(*rotate3d(*p)) }
      lx = (top_pts.sum { |p| p[0] } / 4.0).to_i
      ly = (top_pts.sum { |p| p[1] } / 4.0 - 10).to_i
      draw_label(img, lx, ly, format("%.1f", b[:val]),
                 gd_color(40, 40, 40), gd_color(250, 250, 248))
    end

    img
  end

  # ── Axis lines with tick marks ────────────────────────────────
  def draw_axes(img, x0_base, n, gap, bar_w, vmax)
    axis_color = gd_color(120, 120, 118)
    tick_color = gd_color(160, 160, 158)

    # X axis — along bar centers at y=front, z=0
    x_start = x0_base - bar_w
    x_end   = x0_base + (n - 1) * gap + bar_w
    y_front = -0.7
    p1 = project2d(*rotate3d(x_start, y_front, 0))
    p2 = project2d(*rotate3d(x_end,   y_front, 0))
    safe_line(img, p1, p2, axis_color)

    # Z axis — vertical at left of chart
    z_top = 8.5
    pz1 = project2d(*rotate3d(x_start, y_front, 0))
    pz2 = project2d(*rotate3d(x_start, y_front, z_top))
    safe_line(img, pz1, pz2, axis_color)

    # Z tick marks (5 ticks)
    5.times do |t|
      zv  = t.to_f / 4 * z_top
      tp1 = project2d(*rotate3d(x_start,       y_front, zv))
      tp2 = project2d(*rotate3d(x_start - 0.2, y_front, zv))
      safe_line(img, tp1, tp2, tick_color)
      val = (zv / z_top * vmax).round(1)
      draw_label(img, tp2[0] - 2, tp2[1], val.to_s,
                 gd_color(80, 80, 80), gd_color(250, 250, 248))
    end

    # Y axis — depth line
    p_y1 = project2d(*rotate3d(x_start, y_front, 0))
    p_y2 = project2d(*rotate3d(x_start,  0.7,    0))
    safe_line(img, p_y1, p_y2, axis_color)
  end

  # ── Tiny pixel-drawn label (no font file needed) ──────────────
  # Renders digits + dot using a 3×5 pixel bitmap font
  def draw_label(img, x, y, text, fg, _bg)
    PIXEL_FONT.each_char.with_index do |chr, ci|
      bits = PIXEL_FONT_GLYPHS[chr]
      next unless bits
      ox = x + ci * 5
      5.times do |row|
        3.times do |col|
          if bits[row * 3 + col] == 1
            img.set_pixel(ox + col, y + row, fg) rescue nil
          end
        end
      end
    end
  end

  # 3×5 bitmap font — digits and decimal point
  PIXEL_FONT_GLYPHS = {
    '0' => [1,1,1, 1,0,1, 1,0,1, 1,0,1, 1,1,1],
    '1' => [0,1,0, 1,1,0, 0,1,0, 0,1,0, 1,1,1],
    '2' => [1,1,1, 0,0,1, 1,1,1, 1,0,0, 1,1,1],
    '3' => [1,1,1, 0,0,1, 0,1,1, 0,0,1, 1,1,1],
    '4' => [1,0,1, 1,0,1, 1,1,1, 0,0,1, 0,0,1],
    '5' => [1,1,1, 1,0,0, 1,1,1, 0,0,1, 1,1,1],
    '6' => [1,1,1, 1,0,0, 1,1,1, 1,0,1, 1,1,1],
    '7' => [1,1,1, 0,0,1, 0,1,0, 0,1,0, 0,1,0],
    '8' => [1,1,1, 1,0,1, 1,1,1, 1,0,1, 1,1,1],
    '9' => [1,1,1, 1,0,1, 1,1,1, 0,0,1, 1,1,1],
    '.' => [0,0,0, 0,0,0, 0,0,0, 0,0,0, 0,1,0],
    ' ' => [0,0,0, 0,0,0, 0,0,0, 0,0,0, 0,0,0],
  }.freeze
  PIXEL_FONT = ''.freeze   # placeholder — text passed directly to draw_label

  # Override draw_label to iterate over the actual string chars
  def draw_label(img, x, y, text, fg, _bg)
    text.each_char.with_index do |chr, ci|
      bits = PIXEL_FONT_GLYPHS[chr]
      next unless bits
      ox = x + ci * 5
      5.times do |row|
        3.times do |col|
          img.set_pixel(ox + col, y + row, fg) rescue nil if bits[row * 3 + col] == 1
        end
      end
    end
  end

  # ════════════════════════════════════════════════════════════
  #  3D MATH
  # ════════════════════════════════════════════════════════════
  def rotate3d(x, y, z)
    x1 =  x * @cos_y + y * @sin_y
    y1 = -x * @sin_y + y * @cos_y
    z1 =  z
    x2 =  x1
    y2 =  y1 * @cos_p - z1 * @sin_p
    z2 =  y1 * @sin_p + z1 * @cos_p
    [x2, y2, z2]
  end

  def project2d(x, y, z)
    denom = (@cam_dist - z).abs
    denom = 0.001 if denom < 0.001
    sx = (@fov * x / denom + @width  / 2.0).to_i
    sy = (-@fov * y / denom + @height / 2.0).to_i
    [sx, sy]
  end

  def sample_grid
    n    = @resolution
    grid = Array.new(n + 1) { Array.new(n + 1) }
    (n + 1).times do |i|
      (n + 1).times do |j|
        x = @xmin + i.to_f / n * (@xmax - @xmin)
        y = @ymin + j.to_f / n * (@ymax - @ymin)
        grid[i][j] = SafeMath.evaluate(@expr, x: x, y: y)
      end
    end
    grid
  end

  # ════════════════════════════════════════════════════════════
  #  COLOR HELPERS
  # ════════════════════════════════════════════════════════════
  def colormap(t)
    t     = t.clamp(0.0, 1.0)
    stops = VIRIDIS
    idx   = t * (stops.size - 1)
    lo    = idx.floor.clamp(0, stops.size - 2)
    hi    = lo + 1
    frac  = idx - lo
    lo_c  = stops[lo]; hi_c = stops[hi]
    [0, 1, 2].map { |k| (lo_c[k] + frac * (hi_c[k] - lo_c[k])).round.clamp(0, 255) }
  end

  def shade_color(rgb, factor)
    rgb.map { |c| (c * factor).round.clamp(0, 255) }
  end

  def darken(rgb, factor) = shade_color(rgb, factor)

  def gd_color(r, g, b)
    GD::Color.rgb(r, g, b)
  end

  # ════════════════════════════════════════════════════════════
  #  DRAWING HELPERS
  # ════════════════════════════════════════════════════════════
  def new_image
    img = GD::Image.new(@width, @height)
    img.filled_rectangle(0, 0, @width - 1, @height - 1, gd_color(*@bg))
    img
  end

  def draw_filled_polygon(img, pts2d, color)
    return if pts2d.size < 3
    img.filled_polygon(pts2d, color)
  rescue StandardError
    nil
  end

  def draw_polygon_outline(img, pts2d, color)
    return if pts2d.size < 2
    pts2d.each_cons(2) { |a, b| safe_line(img, a, b, color) }
    safe_line(img, pts2d.last, pts2d.first, color)
  end

  def safe_line(img, a, b, color)
    x1 = a[0].clamp(0, @width  - 1)
    y1 = a[1].clamp(0, @height - 1)
    x2 = b[0].clamp(0, @width  - 1)
    y2 = b[1].clamp(0, @height - 1)
    img.line(x1, y1, x2, y2, color)
  rescue StandardError
    nil
  end
end

(irb):51: warning: already initialized constant #<Class:0x00007f4432103d58>::Plot3D::VIRIDIS
(irb):51: warning: previous definition of VIRIDIS was here
(irb):58: warning: already initialized constant #<Class:0x00007f4432103d58>::Plot3D::BAR_PALETTE
(irb):58: warning: previous definition of BAR_PALETTE was here
(irb):312: warning: already initialized constant #<Class:0x00007f4432103d58>::Plot3D::PIXEL_FONT_GLYPHS
(irb):312: warning: previous definition of PIXEL_FONT_GLYPHS was here
(irb):326: warning: already initialized constant #<Class:0x00007f4432103d58>::Plot3D::PIXEL_FONT
(irb):326: warning: previous definition of PIXEL_FONT was here


:safe_line

In [5]:


# ═════════════════════════════════════════════════════════════════
#  CELL 1 — Ripple:  z = sin(sqrt(x² + y²))
# ═════════════════════════════════════════════════════════════════
name = "ripple.png"
Plot3D.new(resolution: 50, yaw: 35, pitch: 28, fov: 460)
  .surface("sin(sqrt(x**2 + y**2))")
  .render(name)
render(name)

# ═════════════════════════════════════════════════════════════════
#  CELL 2 — Saddle:  z = x² - y²
# ═════════════════════════════════════════════════════════════════
name = "saddle.png"
Plot3D.new(resolution: 40, yaw: 40, pitch: 25, fov: 460,
           xmin: -3, xmax: 3, ymin: -3, ymax: 3)
  .surface("x**2 - y**2")
  .render(name)
render(name)

# ═════════════════════════════════════════════════════════════════
#  CELL 3 — Wave:  z = cos(x) * sin(y)
# ═════════════════════════════════════════════════════════════════
name = "wave.png"
Plot3D.new(resolution: 45, yaw: 30, pitch: 30, fov: 460,
           xmin: -4, xmax: 4, ymin: -4, ymax: 4)
  .surface("cos(x) * sin(y)")
  .render(name)
render(name)

# ═════════════════════════════════════════════════════════════════
#  CELL 4 — Cone:  z = sqrt(x² + y²)
# ═════════════════════════════════════════════════════════════════
name = "cone.png"
Plot3D.new(resolution: 45, yaw: 35, pitch: 25, fov: 460,
           xmin: -4, xmax: 4, ymin: -4, ymax: 4, zmax: 5)
  .surface("sqrt(x**2 + y**2)")
  .render(name)
render(name)

# ═════════════════════════════════════════════════════════════════
#  CELL 5 — Paraboloid:  z = x² + y²
# ═════════════════════════════════════════════════════════════════
name = "paraboloid.png"
Plot3D.new(resolution: 40, yaw: 35, pitch: 28, fov: 460,
           xmin: -3, xmax: 3, ymin: -3, ymax: 3, zmax: 9)
  .surface("x**2 + y**2")
  .render(name)
render(name)

# ═════════════════════════════════════════════════════════════════
#  CELL 6 — 3D Bar chart  (fixed: bigger, axes, value labels)
# ═════════════════════════════════════════════════════════════════
data = {
  "Ruby"   => 4.2,
  "Python" => 9.1,
  "Julia"  => 6.7,
  "R"      => 5.8,
  "Rust"   => 3.9,
}

name = "bars.png"
Plot3D.new(
  width:    700,
  height:   520,
  xmin:    -4.0, xmax: 4.0,
  ymin:    -2.0, ymax: 2.0,
  yaw:      22.0,
  pitch:    28.0,
  fov:      380.0,
  cam_dist: 16.0
).bars(data).render(name)
render(name)

"<img src='ripple.png' />"

"<img src='saddle.png' />"

"<img src='wave.png' />"

"<img src='cone.png' />"

"<img src='paraboloid.png' />"

"<img src='bars.png' />"